# 05. Universe Selection & Strategy Construction

## 📋 개요
모델의 예측 결과를 바탕으로 **수익률 중심의 투자 후보군(Universe)**을 선정합니다.
기계적인 점수 합산보다는, **"최고 기대 수익률 종목을 우선 노출하되, 리스크와 정확도 정보를 함께 제공하여 사용자가 최종 결정"**하는 Human-in-the-loop 방식을 채택합니다.

## ✨ 핵심 프로세스
### 1. 이중 날짜 기준 (Dual-Date Evaluation)
- **`model_train_date` (과거)**: 모델의 정확도(Accuracy) 검증용
- **`forecast_date` (미래)**: 기대 수익률(Return) 및 미래 리스크 측정용

### 2. 전략 구성 방법론: "Return-First, Risk-Aware"
1. **Hard Filtering (안전장치)**
   - 거래정지, 상장폐지, 초저유동성, 작전주 혐의 종목 등 "투자 불가/위험" 종목을 물리적으로 제거합니다.
2. **Profitability Ranking (수익 추구)**
   - 미래 예측 경로에서 산출된 **`daily_log_return` (시간당 기대 수익률)**을 기준으로 내림차순 정렬합니다.
3. **Informative Metrics (정보 제공)**
   - 단순 순위 외에 **방향성 정확도(Directional Accuracy)**와 **복합 리스크 점수**를 함께 제공하여, 사용자가 "수익은 높지만 너무 위험한 종목"을 걸러낼 수 있도록 돕습니다.

## 🔄 데이터 흐름
```text
03단계 예측 결과 (predictions.parquet)
    ↓
[정확도 평가] (과거 데이터 기반 신뢰도 측정)
    ↓
04단계 미래 예측 (forecasts.parquet)
    ↓
[수익성 평가] (최적 매매 타이밍 및 수익률 산출)
    ↓
[리스크 평가] (변동성, MDD, 작전주 탐지)
    ↓
[Hard Filter 적용] (부적격 종목 제거)
    ↓
[수익률 기준 정렬 및 리포트 생성]
    ↓
최종 투자 후보 (universe_candidates.parquet / investment_report.csv)

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

from src.utils.config import load_config
from src.universe.select_universe import (
    build_production_universe,
    select_candidates_from_scores
)

warnings.filterwarnings('ignore')

## 1️⃣ 설정 및 날짜 기준 정의

In [ ]:
# ==========================================
# 설정 로드
# ==========================================
cfg = load_config()
ref_date = cfg['project']['reference_date']

# ==========================================
# 이중 날짜 기준 설정 ⚠️ 핵심
# ==========================================
MODEL_DATE = cfg['universe']['model_date']  # 예: "2026-01-20" (학습 기준일)
model_date_dt = pd.to_datetime(MODEL_DATE)

print(f"📅 날짜 기준 설정:")
print(f"   - 모델 학습 기준일 (참값 존재): {MODEL_DATE}")

# ==========================================
# 경로 설정
# ==========================================
result_dir = Path(cfg['paths']['result_dir']) / ref_date
processed_dir = Path(cfg['paths']['processed_dir']) / ref_date
output_dir = result_dir / 'universe'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📁 경로:")
print(f"   - 예측 결과: {result_dir / 'forecasts'}")
print(f"   - Universe 출력: {output_dir}")

## 2️⃣ 데이터 로드

In [ ]:
# ==========================================
# 1. 과거 예측 결과 (정확도 평가용)
# ==========================================
print("\n📥 과거 예측 결과 로드 (정확도 평가용)...")
df_past_pred = pd.read_parquet(Path(cfg['paths']['result_dir']) / MODEL_DATE / 'predictions.parquet')
df_past_pred['date'] = pd.to_datetime(df_past_pred['date'])

# 학습 기간 데이터만 필터링 (참값 존재)
df_past_pred = df_past_pred[df_past_pred['date'] <= MODEL_DATE].copy()

print(f"   - 총 행수: {len(df_past_pred):,}")
print(f"   - 기간: {df_past_pred['date'].min()} ~ {df_past_pred['date'].max()}")
print(f"   - 종목 수: {df_past_pred['ticker'].nunique()}")

# ==========================================
# 2. 미래 예측 결과 (수익성 평가용)
# ==========================================
print("\n📥 미래 예측 결과 로드 (수익성 평가용)...")
forecast_path = result_dir / 'forecasts' / 'future_forecasts.parquet'

if not forecast_path.exists():
    raise FileNotFoundError(
        f"❌ 미래 예측 파일을 찾을 수 없습니다: {forecast_path}\n"
        "💡 Tip: 04_forecast_future.ipynb를 먼저 실행하세요."
    )

df_future = pd.read_parquet(forecast_path)
df_future['date'] = pd.to_datetime(df_future['date'])

print(f"   - 총 행수: {len(df_future):,}")
print(f"   - 기간: {df_future['date'].min()} ~ {df_future['date'].max()}")
print(f"   - 종목 수: {df_future['ticker'].nunique()}")

# ==========================================
# 3. Feature 데이터셋 (리스크 메타 정보)
# ==========================================
print("\n📥 리스크 메타 데이터 로드...")
df_meta = pd.read_parquet(processed_dir / 'dataset.parquet')
df_meta['date'] = pd.to_datetime(df_meta['date'])

# 최신 날짜의 메타 정보만 사용
latest_meta_date = df_meta['date'].max()
df_meta_latest = df_meta[df_meta['date'] == latest_meta_date].copy()

print(f"   - 기준일: {latest_meta_date}")
print(f"   - 종목 수: {df_meta_latest['ticker'].nunique()}")

## 3️⃣ 정확도 평가 (Accuracy Scoring)

### 🔬 평가 지표 정의
모델이 과거 데이터를 얼마나 잘 맞췄는지 평가하여, 미래 예측의 **신뢰도(Confidence)**를 측정합니다.

#### 1. 방향성 정확도 (Directional Accuracy)
- **개념**: "오른다/내린다"의 방향을 맞춘 비율입니다.
- **의의**: 수익률의 크기보다 **방향**이 중요한 트레이딩 관점에서 가장 직관적인 지표입니다.
- **기준**: 50%는 랜덤, 60% 이상이면 유의미한 예측 능력으로 간주합니다.

#### 2. RMSE 기반 신뢰도 (Confidence Score)
- **개념**: 예측 오차(RMSE)의 역수 (`1 / (1 + RMSE)`)입니다.
- **의의**: 오차의 절대적인 크기가 작을수록 높은 점수를 부여합니다.

**전략 적용**:
이 지표들은 필터링의 절대 기준보다는, 수익률 상위 종목 중 **"믿을 수 있는 종목"**을 선별하는 참고 지표로 활용됩니다.

In [ ]:
print("\n" + "="*65)
print("3️⃣ 정확도 평가")
print("="*65)

# ==========================================
# model_date 이전 데이터 필터링
# ==========================================
df_past_eval = df_past_pred[df_past_pred['date'] <= model_date_dt].copy()

if len(df_past_eval) == 0:
    raise ValueError(
        f"❌ {MODEL_DATE} 이전의 예측 데이터가 없습니다.\n"
        f"   03단계 predictions.parquet의 날짜 범위를 확인하세요."
    )

print(f"\n   - 평가 데이터 행수: {len(df_past_eval):,}")
print(f"   - 평가 기간: {df_past_eval['date'].min()} ~ {df_past_eval['date'].max()}")

# ==========================================
# 종목별 정확도 지표 계산
# ==========================================
print("\n🎯 정확도 지표 계산 중...")
print("   - Method 2: 방향성 정확도 (상승/하락 방향 적중률)")
print("   - Method 3: RMSE 역수 기반 (오차 크기 신뢰도)")

accuracy_metrics = []

for ticker in tqdm(df_past_eval['ticker'].unique(), desc="정확도 계산"):
    ticker_data = df_past_eval[df_past_eval['ticker'] == ticker].copy()
    
    # 모든 Horizon의 예측 결과 통합
    all_errors = []
    direction_matches = []  # 방향성 일치 여부
    
    for h in range(1, 6):  # h1~h5
        pred_col = f'pred_target_log_close_h{h}'
        true_col = f'true_target_log_close_h{h}'
        
        if pred_col in ticker_data.columns and true_col in ticker_data.columns:
            valid_rows = ticker_data[[pred_col, true_col]].dropna()
            
            if len(valid_rows) > 0:
                # 오차 (RMSE용)
                errors = valid_rows[pred_col] - valid_rows[true_col]
                all_errors.extend(errors.values)
                
                # 방향성 일치 (전일 대비 상승/하락)
                pred_change = valid_rows[pred_col].diff()
                true_change = valid_rows[true_col].diff()
                direction_match = (pred_change * true_change) > 0
                direction_matches.extend(direction_match.dropna().values)
    
    if len(all_errors) > 10:  # 최소 10개 이상 예측값 필요
        all_errors = np.array(all_errors)
        
        # RMSE
        rmse = np.sqrt(np.mean(all_errors ** 2))
        mae = np.mean(np.abs(all_errors))
        
        # Method 2: Directional Accuracy
        if len(direction_matches) > 0:
            directional_accuracy = np.mean(direction_matches)
        else:
            directional_accuracy = 0.5  # 기본값 (랜덤 수준)
        
        # Method 3: RMSE 역수 기반 신뢰도
        confidence_rmse = 1 / (1 + rmse)
        
        accuracy_metrics.append({
            'ticker': ticker,
            'rmse': rmse,
            'mae': mae,
            'directional_accuracy': directional_accuracy,
            'confidence_rmse': confidence_rmse,
            'num_predictions': len(all_errors)
        })

df_accuracy = pd.DataFrame(accuracy_metrics)

# ==========================================
# Method 2: 방향성 정확도 점수
# ==========================================
# 0.5(랜덤) ~ 1.0(완벽) 범위를 0~1로 재조정
df_accuracy['accuracy_score_directional'] = (
    (df_accuracy['directional_accuracy'] - 0.5) * 2
).clip(0, 1)

# ==========================================
# Method 3: RMSE 역수 기반 점수
# ==========================================
df_accuracy['accuracy_score_confidence'] = df_accuracy['confidence_rmse']

# 순위 부여
df_accuracy['accuracy_rank'] = df_accuracy['rmse'].rank()

# ==========================================
# 결과 요약
# ==========================================
print(f"\n✅ 정확도 평가 완료")
print(f"   - 평가 종목 수: {len(df_accuracy)}")

print(f"\n[RMSE 통계]")
print(f"   - 평균 RMSE: {df_accuracy['rmse'].mean():.4f}")
print(f"   - 중앙값 RMSE: {df_accuracy['rmse'].median():.4f}")
print(f"   - 최소 RMSE: {df_accuracy['rmse'].min():.4f}")

print(f"\n[Method 2 - 방향성 정확도]")
print(f"   - 평균: {df_accuracy['directional_accuracy'].mean():.2%}")
print(f"   - 중앙값: {df_accuracy['directional_accuracy'].median():.2%}")
print(f"   - 60% 이상: {(df_accuracy['directional_accuracy'] >= 0.6).sum()}개")

print(f"\n[Method 3 - RMSE 역수 신뢰도]")
print(f"   - 평균: {df_accuracy['confidence_rmse'].mean():.4f}")
print(f"   - 중앙값: {df_accuracy['confidence_rmse'].median():.4f}")

# 상위 10개 확인
print(f"\n[Top 10 - 방향성 정확도 기준]")
display(df_accuracy.nlargest(10, 'directional_accuracy')[[
    'ticker', 'directional_accuracy', 'confidence_rmse', 'rmse', 'num_predictions'
]])

print(f"\n[Top 10 - RMSE 역수 신뢰도 기준]")
display(df_accuracy.nlargest(10, 'confidence_rmse')[[
    'ticker', 'confidence_rmse', 'directional_accuracy', 'rmse', 'num_predictions'
]])

## 4️⃣ 수익성 평가 (Return Scoring)

### 📐 핵심 지표: 시간당 로그 수익률 (Daily Log Return)

**핵심 수식**:
```
일평균 로그 수익률 = [log(매도가) - log(매수가)] / 보유기간
```

**이론적 배경**:
- 주가는 기하 브라운 운동: `dX/dt = rX`
- 로그 수익률의 시간 가산성 (Log Returns are Time-Additive)
- 자본 효율성: 5일 만에 10% 오르는 거래가, 20일 만에 12% 오르는 거래보다 자본 회전율 측면에서 유리

**제약 조건**:
- **최소 보유 기간**: 5일 (단기 매매 지양)
- **탐색 범위**: 전체 예측 기간 (완전 탐색)

**알고리즘**: 
- NumPy 벡터화 완전 탐색 O(N²)

👉 **이 지표(`daily_log_return`)가 최종 Universe의 정렬(Ranking) 기준이 됩니다.**

In [ ]:
# ==========================================
# Trading 유틸리티 임포트
# ==========================================
from src.utils.trading import find_best_trade_vectorized

print("✅ Trading 유틸리티 로드 완료")

In [ ]:
# ==========================================
# 종목별 최적 수익률 계산
# ==========================================
print("\n💰 수익성 평가 중 (시간당 로그 수익률 기준)...")

MIN_HOLD_DAYS = 5  # 최소 보유 기간 (1주일)

return_metrics = []
failed_tickers = []

for ticker in tqdm(df_future['ticker'].unique(), desc="최적 수익률 계산"):
    try:
        # 종목별 예측 데이터 추출 (날짜 정렬)
        ticker_data = df_future[
            df_future['ticker'] == ticker
        ].sort_values('date').reset_index(drop=True)
        
        if len(ticker_data) < MIN_HOLD_DAYS:
            failed_tickers.append((ticker, f"데이터 부족 ({len(ticker_data)}일)"))
            continue
        
        # ⭐ 핵심: pred_log_close 직접 사용 (이미 로그 변환됨)
        log_prices = ticker_data['pred_log_close'].values
        
        # 최적 매매 시점 탐색
        buy_idx, sell_idx, daily_log_return, hold_days = find_best_trade_vectorized(
            log_prices, min_hold=MIN_HOLD_DAYS
        )
        
        if np.isnan(daily_log_return) or np.isinf(daily_log_return):
            failed_tickers.append((ticker, "유효한 거래 없음"))
            continue
        
        # 매수/매도 시점 정보
        buy_date = ticker_data.iloc[buy_idx]['date']
        sell_date = ticker_data.iloc[sell_idx]['date']
        buy_price = ticker_data.iloc[buy_idx]['pred_close']
        sell_price = ticker_data.iloc[sell_idx]['pred_close']
        
        # 총 수익률 (검증용)
        total_log_return = log_prices[sell_idx] - log_prices[buy_idx]
        total_return_pct = (np.exp(total_log_return) - 1) * 100
        
        # 연율화 수익률 (참고용, 252 영업일 기준)
        annualized_return = daily_log_return * 252
        
        return_metrics.append({
            'ticker': ticker,
            'daily_log_return': daily_log_return,  # ⭐ 핵심 지표
            'total_log_return': total_log_return,
            'total_return_pct': total_return_pct,
            'annualized_return': annualized_return,
            'hold_days': hold_days,
            'buy_date': buy_date,
            'sell_date': sell_date,
            'buy_price': buy_price,
            'sell_price': sell_price,
            'price_change_pct': (sell_price / buy_price - 1) * 100
        })
    
    except Exception as e:
        failed_tickers.append((ticker, f"오류: {str(e)}"))
        continue

df_return = pd.DataFrame(return_metrics)

# ==========================================
# 점수화 (Min-Max 정규화)
# ==========================================

# [0, 1] 정규화
min_return = df_return['daily_log_return'].min()
max_return = df_return['daily_log_return'].max()

df_return['return_score'] = (
    (df_return['daily_log_return'] - min_return) / (max_return - min_return)
)

# 순위
df_return['return_rank'] = df_return['daily_log_return'].rank(
    ascending=False, method='min'
)

# ==========================================
# 결과 요약
# ==========================================
print(f"\n{'='*65}")
print("✅ 수익성 평가 완료")
print(f"{'='*65}")
print(f"   - 성공 종목 수: {len(df_return):,}")
print(f"   - 실패 종목 수: {len(failed_tickers):,}")

print(f"\n[시간당 로그 수익률 통계]")
print(f"   - 평균: {df_return['daily_log_return'].mean():.6f}")
print(f"   - 중앙값: {df_return['daily_log_return'].median():.6f}")
print(f"   - 표준편차: {df_return['daily_log_return'].std():.6f}")
print(f"   - 최소값: {df_return['daily_log_return'].min():.6f}")
print(f"   - 최대값: {df_return['daily_log_return'].max():.6f}")

print(f"\n[총 수익률 통계]")
print(f"   - 평균: {df_return['total_return_pct'].mean():.2f}%")
print(f"   - 중앙값: {df_return['total_return_pct'].median():.2f}%")

print(f"\n[보유 기간 통계]")
print(f"   - 평균: {df_return['hold_days'].mean():.1f}일")
print(f"   - 중앙값: {df_return['hold_days'].median():.0f}일")

# 수익률 분포
positive_returns = (df_return['total_return_pct'] > 0).sum()
print(f"\n[수익률 분포]")
print(f"   - 양수 수익: {positive_returns}/{len(df_return)} "
      f"({positive_returns/len(df_return)*100:.1f}%)")

print(f"\n[Top 10 종목 - 시간당 로그 수익률 기준]")
display(df_return.nlargest(10, 'daily_log_return')[[
    'ticker', 'return_rank', 'daily_log_return', 'total_return_pct',
    'hold_days', 'buy_date', 'sell_date'
]])

# 실패 종목 확인
if len(failed_tickers) > 0:
    print(f"\n⚠️  실패 종목 샘플 (처음 5개):")
    for ticker, reason in failed_tickers[:5]:
        print(f"   - {ticker}: {reason}")

## 5️⃣ 위험도 평가 (Risk Scoring)

### 📐 종목 내재 위험 (Aleatoric Uncertainty)

**정의**: 모델 예측 오차와 무관한, 종목 자체의 불확정성

**vs 정확도**: 
- **정확도(Accuracy)**: "모델이 얼마나 잘 맞추는가?" (Epistemic)
- **위험도(Risk)**: "이 종목 자체가 얼마나 예측 불가능한가?" (Aleatoric)

### 🔍 5대 표준 지표

| 지표 | 의미 | 해석 |
|------|------|------|
| **Volatility** | 변동성 (기본) | 높을수록 불안정 |
| **Downside Risk** | 하방 위험 | 손실만 측정 (상승은 OK) |
| **VaR / CVaR** | 극단 리스크 | 최악 5% 평균 손실 |
| **Max Drawdown** | 최대 낙폭 | 고점 대비 최대 손실률 |
| **Skew / Kurt** | 분포 형태 | 비대칭성, Fat Tail |

**데이터 소스**: 예측값 시계열 (`pred_log_close`)  
→ "미래 위험도" 평가

**Why 예측값?**
- 과거 위험 ≠ 미래 위험 (시장 국면 변화)
- 모델이 예측한 미래 궤적의 불확정성 측정

In [ ]:
# ==========================================
# Risk 유틸리티 임포트
# ==========================================
from src.utils.risk import (
    calculate_risk_metrics,
    calculate_composite_risk_score,
    normalize_risk_scores
)

print("✅ Risk 유틸리티 로드 완료")

In [ ]:
# ==========================================
# 종목별 위험 지표 계산
# ==========================================
print("\n⚠️  위험도 평가 중 (5대 표준 지표)...")

risk_results = []
failed_risk_tickers = []

for ticker in tqdm(df_future['ticker'].unique(), desc="위험 지표 계산"):
    try:
        # 종목별 예측 데이터 추출 (날짜 정렬)
        ticker_data = df_future[
            df_future['ticker'] == ticker
        ].sort_values('date').reset_index(drop=True)
        
        if len(ticker_data) < 5:
            failed_risk_tickers.append((ticker, f"데이터 부족 ({len(ticker_data)}일)"))
            continue
        
        # ⭐ 핵심: 예측값 시계열 사용 (미래 위험도)
        log_prices = ticker_data['pred_log_close'].values
        
        # 5대 위험 지표 계산
        metrics = calculate_risk_metrics(
            log_prices,
            is_log_prices=True  # 이미 로그 변환됨
        )
        
        # NaN 체크
        if any(np.isnan(v) for v in metrics.values()):
            failed_risk_tickers.append((ticker, "NaN 발생"))
            continue
        
        # ⭐ 복합 리스크 스코어 계산 (calculate_composite_risk_score 사용)
        composite_score = calculate_composite_risk_score(metrics)
        
        # 결과 저장 (모든 지표 + 복합 점수)
        result = {
            'ticker': ticker,
            **metrics,  # volatility, downside_risk, var_95, cvar_95, max_drawdown, skewness, kurtosis
            'risk_composite_raw': composite_score  # ⭐ 이 컬럼이 필요함
        }
        risk_results.append(result)
    
    except Exception as e:
        failed_risk_tickers.append((ticker, f"오류: {str(e)}"))
        continue

df_risk = pd.DataFrame(risk_results)

# ==========================================
# 정규화 및 순위 계산
# ==========================================
df_risk = normalize_risk_scores(df_risk, score_col='risk_composite_raw')

# Safety Score (역순: 높을수록 안전)
df_risk['safety_score'] = 1 - df_risk['risk_score_normalized']

# ==========================================
# 02단계 메타 데이터 병합 (유동성, 거래정지 플래그)
# ==========================================
df_risk = df_risk.merge(
    df_meta_latest[['ticker', 'liquidity_score', 'is_suspended', 'is_delisted']],
    on='ticker',
    how='left'
)

# 결측값 처리
df_risk['liquidity_score'] = df_risk['liquidity_score'].fillna(0)
df_risk['is_suspended'] = df_risk['is_suspended'].fillna(0).astype(int)
df_risk['is_delisted'] = df_risk['is_delisted'].fillna(0).astype(int)

# ==========================================
# 결과 요약
# ==========================================
print(f"\n{'='*65}")
print("✅ 위험도 평가 완료")
print(f"{'='*65}")
print(f"   - 성공 종목 수: {len(df_risk):,}")
print(f"   - 실패 종목 수: {len(failed_risk_tickers):,}")

print(f"\n[1. Volatility (변동성)]")
print(f"   - 평균: {df_risk['volatility'].mean():.6f}")
print(f"   - 중앙값: {df_risk['volatility'].median():.6f}")
print(f"   - 연율화: {df_risk['volatility'].mean() * np.sqrt(252):.2%}")

print(f"\n[2. Downside Risk (하방 위험)]")
print(f"   - 평균: {df_risk['downside_risk'].mean():.6f}")
downside_ratio = df_risk['downside_risk'].mean() / df_risk['volatility'].mean()
print(f"   - Downside/Total 비율: {downside_ratio:.2%}")

print(f"\n[3. VaR/CVaR (극단 리스크, 95% 신뢰)]")
print(f"   - VaR (5th percentile): {df_risk['var_95'].mean():.6f}")
print(f"   - CVaR (Expected Shortfall): {df_risk['cvar_95'].mean():.6f}")
print(f"   - CVaR 연율화: {df_risk['cvar_95'].mean() * 252:.2%}")

print(f"\n[4. Maximum Drawdown (최대 낙폭)]")
print(f"   - 평균 MDD: {df_risk['max_drawdown'].mean():.2%}")
print(f"   - 중앙값 MDD: {df_risk['max_drawdown'].median():.2%}")
print(f"   - 최악 MDD: {df_risk['max_drawdown'].min():.2%}")

print(f"\n[5. Skewness/Kurtosis (분포 형태)]")
avg_skew = df_risk['skewness'].mean()
print(f"   - 평균 Skewness: {avg_skew:.4f} ", end="")
print(f"({'하락 쏠림' if avg_skew < 0 else '상승 쏠림'})")
avg_kurt = df_risk['kurtosis'].mean()
print(f"   - 평균 Excess Kurtosis: {avg_kurt:.4f} ", end="")
print(f"({'Fat Tail' if avg_kurt > 0 else 'Thin Tail'})")

print(f"\n[복합 리스크 스코어]")
print(f"   - 평균 (원점수): {df_risk['risk_composite_raw'].mean():.6f}")
print(f"   - 평균 (정규화): {df_risk['risk_score_normalized'].mean():.4f}")
print(f"   - 평균 안전 점수: {df_risk['safety_score'].mean():.4f}")

# ==========================================
# 안전한 종목 Top 10 (낮은 위험)
# ==========================================
print(f"\n[Top 10 안전 종목 - 낮은 위험 기준]")
display(df_risk.nsmallest(10, 'risk_composite_raw')[[
    'ticker', 'risk_rank', 'volatility', 'downside_risk',
    'cvar_95', 'max_drawdown', 'kurtosis', 'safety_score'
]])

# ==========================================
# 위험한 종목 Top 10 (높은 위험)
# ==========================================
print(f"\n[Top 10 위험 종목 - 높은 위험 기준]")
display(df_risk.nlargest(10, 'risk_composite_raw')[[
    'ticker', 'risk_rank', 'volatility', 'downside_risk',
    'cvar_95', 'max_drawdown', 'kurtosis', 'safety_score'
]])

# 실패 종목 샘플
if len(failed_risk_tickers) > 0:
    print(f"\n⚠️  실패 종목 샘플 (처음 5개):")
    for ticker, reason in failed_risk_tickers[:5]:
        print(f"   - {ticker}: {reason}")

In [ ]:
# ==========================================
# 위험 지표 간 상관관계 분석 (선택)
# ==========================================
print("\n[위험 지표 간 상관관계]")

risk_cols = ['volatility', 'downside_risk', 'var_95', 'cvar_95', 'max_drawdown', 'kurtosis']
correlation = df_risk[risk_cols].corr()

display(correlation.round(3))

# 해석 가이드
print("\n💡 해석 가이드:")
print("   - Volatility vs Downside Risk: 높으면 손실 변동성 우세")
print("   - CVaR vs MDD: 높으면 극단 리스크 연관성 강함")
print("   - Excess Kurtosis vs 타 지표: 높으면 Fat Tail이 주요 위험 요인")

## 5-1️⃣ 하드 필터링 (Hard Filters)

### 🚧 투자 부적격 종목 물리적 제거 (Safety First)

기대 수익률이 높아도, 거래가 불가능하거나 리스크가 허용 범위를 넘어서는 종목은 후보군에서 배제

| 필터링 항목 | 제거 기준 | 근거 |
|:---:|:---|:---|
| **1. 거래정지/상폐** | `is_suspended=1` or `is_delisted=1` | 매매 불가능 |
| **2. 작전주/테마주** | 20일 +100% 급등 AND 거래량 5배 폭증 | 인위적 시세 조종 위험 (설계서 반영) |
| **3. 동전주 (Penny)** | 평균 예측가 < 1,000원 | 극심한 변동성 및 관리종목 지정 위험 |
| **4. 초저유동성** | 20일 평균 거래대금 < 5천만 원 | 슬리피지 및 체결 불가 위험 |
| **5. 고위험군** | 복합 리스크 점수(Risk Composite) > 0.8 | 감당 불가능한 하방 위험 |

**철학**: 
> "학습은 전부, 투자 후보는 엄선해서"

In [ ]:
# ==========================================
# Filter 유틸리티 임포트
# ==========================================
from src.utils.filters import (
    apply_hard_filters,
    analyze_filter_impact
)

print("✅ Hard Filter 유틸리티 로드 완료")

## 6️⃣ 수익률 기준 정렬 및 Universe 선정

### 📊 선정 로직: Profit Maximization with Safety Checks

복잡한 가중치 수식 대신, 직관적인 **수익률 우선 순위** 전략을 사용합니다.

1. **지표 통합**: 정확도(과거), 수익성(미래), 위험도(미래), 메타정보를 하나로 병합합니다.
2. **Hard Constraints 적용**: 앞서 정의한 필터링 조건을 적용하여 부실 종목을 탈락시킵니다.
3. **Ranking**: 살아남은 종목들을 **`daily_log_return` (기대 수익률)** 순으로 내림차순 정렬합니다.
4. **Candidate Selection**: 상위 **Top-K (기본 100개)** 종목을 1차 후보군으로 선정합니다.

**사용자 역할**:
- 시스템은 "돈을 가장 잘 벌 것으로 예상되는" 종목들을 순서대로 나열해 줍니다.
- 사용자는 생성된 리포트를 통해 해당 종목의 **정확도(신뢰성)**와 **리스크**를 확인하고 최종 포트폴리오에 편입할지 결정합니다.

In [ ]:
print("\n" + "="*65)
print("6️⃣ 수익률 기준 Universe 선정")
print("="*65)

# ==========================================
# 1. 3대 지표 통합
# ==========================================
print("\n🔗 평가 지표 통합 중...")

# 정확도 + 수익성 + 위험도 통합
df_universe = df_accuracy.merge(df_return, on='ticker', how='inner')
df_universe = df_universe.merge(df_risk, on='ticker', how='inner')

print(f"   - 통합 완료: {len(df_universe):,}개 종목")
print(f"   - 컬럼 수: {len(df_universe.columns)}개")

# 통합 후 컬럼 확인 (디버깅용)
print(f"\n[통합된 주요 컬럼]")
print(f"   - 정확도: directional_accuracy, confidence_rmse, rmse")
print(f"   - 수익성: daily_log_return, total_return_pct, hold_days")
print(f"   - 위험: risk_composite_raw, volatility, max_drawdown")
print(f"   - 메타: liquidity_score, is_suspended, is_delisted")

# ==========================================
# 2. Hard Constraints (필수 조건)
# ==========================================
print("\n🚫 Hard Constraints 적용 중...")

initial_count = len(df_universe)

# 거래 가능 종목만
df_universe = df_universe[
    (df_universe['is_suspended'] == 0) &
    (df_universe['is_delisted'] == 0)
]
suspended_removed = initial_count - len(df_universe)

# 최소 유동성 기준
MIN_LIQUIDITY = 5e7  # 5천만 원
before_liquidity = len(df_universe)
df_universe = df_universe[df_universe['liquidity_score'] >= MIN_LIQUIDITY]
liquidity_removed = before_liquidity - len(df_universe)

# 리스크 상한 (선택: 너무 위험한 종목 제거)
MAX_RISK = 0.8  # 기존 0.7보다 완화 (더 많은 후보 확보)
before_risk = len(df_universe)
df_universe = df_universe[df_universe['risk_composite_raw'] <= MAX_RISK]
risk_removed = before_risk - len(df_universe)

# 정확도 하한 (선택: 너무 부정확한 종목 제거)
# 상위 1000개로 완화 (더 많은 후보 확보)
MAX_ACCURACY_RANK = 1000
before_accuracy = len(df_universe)
df_universe = df_universe[df_universe['accuracy_rank'] <= MAX_ACCURACY_RANK]
accuracy_removed = before_accuracy - len(df_universe)

print(f"\n[필터링 결과]")
print(f"   - 초기 종목: {initial_count:,}개")
print(f"   - 거래정지/상폐 제거: {suspended_removed:,}개")
print(f"   - 저유동성 제거: {liquidity_removed:,}개")
print(f"   - 고위험 제거: {risk_removed:,}개")
print(f"   - 저정확도 제거: {accuracy_removed:,}개")
print(f"   ✅ 남은 종목: {len(df_universe):,}개")

# ==========================================
# 3. 수익률 기준 내림차순 정렬
# ==========================================
print("\n📊 수익률 기준 정렬 중...")

# 핵심 지표: daily_log_return (시간당 로그 수익률)
df_universe = df_universe.sort_values('daily_log_return', ascending=False).reset_index(drop=True)

# 순위 부여
df_universe['return_rank'] = range(1, len(df_universe) + 1)

print(f"   - 정렬 기준: daily_log_return (시간당 로그 수익률)")
print(f"   - 1위 수익률: {df_universe.iloc[0]['daily_log_return']:.6f}")
print(f"   - 중앙값 수익률: {df_universe['daily_log_return'].median():.6f}")
print(f"   - 최소 수익률: {df_universe.iloc[-1]['daily_log_return']:.6f}")

# ==========================================
# 4. 넓은 Top-K 선정
# ==========================================
TOP_K = 100  # 충분히 넓게 (사용자가 직접 선택할 여지)

if len(df_universe) < TOP_K:
    print(f"\n⚠️  필터링 후 종목 수({len(df_universe)})가 TOP_K({TOP_K})보다 적습니다.")
    print(f"   → 전체 {len(df_universe)}개 종목을 후보로 선정합니다.")
    df_candidates = df_universe.copy()
else:
    df_candidates = df_universe.head(TOP_K).copy()

print(f"\n✅ 최종 후보 선정 완료")
print(f"   - 후보 종목 수: {len(df_candidates):,}개")
print(f"   - 선정 기준: 예상 수익률 상위 {min(TOP_K, len(df_universe))}개")

# ==========================================
# 5. 후보군 통계
# ==========================================
print(f"\n📊 후보군 통계:")
print(f"   - 평균 일평균 로그 수익률: {df_candidates['daily_log_return'].mean():.6f}")
print(f"   - 평균 총 수익률: {df_candidates['total_return_pct'].mean():.2f}%")
print(f"   - 평균 보유 기간: {df_candidates['hold_days'].mean():.1f}일")
print(f"   - 평균 방향성 정확도: {df_candidates['directional_accuracy'].mean():.2%}")
print(f"   - 평균 신뢰도(RMSE역수): {df_candidates['confidence_rmse'].mean():.4f}")
print(f"   - 평균 RMSE: {df_candidates['rmse'].mean():.4f}")
print(f"   - 평균 리스크 점수: {df_candidates['risk_composite_raw'].mean():.4f}")
print(f"   - 평균 변동성: {df_candidates['volatility'].mean():.6f}")

# 수익률 분포
positive_returns = (df_candidates['total_return_pct'] > 0).sum()
print(f"\n   - 양수 수익 종목: {positive_returns}/{len(df_candidates)} ({positive_returns/len(df_candidates)*100:.1f}%)")

# 정확도 분포
high_accuracy = (df_candidates['directional_accuracy'] >= 0.6).sum()
print(f"   - 방향성 정확도 60% 이상: {high_accuracy}/{len(df_candidates)} ({high_accuracy/len(df_candidates)*100:.1f}%)")

# 위험도 분포
low_risk = (df_candidates['risk_composite_raw'] <= 0.5).sum()
print(f"   - 리스크 점수 0.5 이하: {low_risk}/{len(df_candidates)} ({low_risk/len(df_candidates)*100:.1f}%)")

## 7️⃣ 사용자 선택을 위한 상세 리포트

In [ ]:
print("\n" + "="*65)
print("7️⃣ 투자 후보 상세 리포트 생성")
print("="*65)

# ==========================================
# 1. ticker_name_map 로드 (종목명 표시)
# ==========================================
try:
    master_path = Path(cfg['paths']['raw_dir']) / ref_date / f"ticker_master_{ref_date}.csv"
    df_master = pd.read_csv(master_path)
    ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
    df_candidates['종목명'] = df_candidates['ticker'].map(ticker_name_map)
    print("✅ ticker_master 로드 완료")
except Exception as e:
    print(f"⚠️  ticker_master 로드 실패: {e}")
    df_candidates['종목명'] = df_candidates['ticker']

# ==========================================
# 2. 리포트용 컬럼 정리 (사용자 의사결정 지원)
# ==========================================

# 가독성을 위한 컬럼명 변경
df_candidates_report = df_candidates.copy()

df_candidates_report['순위'] = df_candidates_report['return_rank']
df_candidates_report['티커'] = df_candidates_report['ticker']

# 수익성 지표
df_candidates_report['예상일평균수익률(로그)'] = df_candidates_report['daily_log_return'].round(6)
df_candidates_report['예상총수익률(%)'] = df_candidates_report['total_return_pct'].round(2)
df_candidates_report['최적보유기간(일)'] = df_candidates_report['hold_days'].astype(int)

# 정확도 지표 (2가지)
df_candidates_report['방향성정확도(%)'] = (
    df_candidates_report['directional_accuracy'] * 100
).round(2)
df_candidates_report['신뢰도(RMSE역수)'] = (
    df_candidates_report['confidence_rmse']
).round(4)
df_candidates_report['RMSE'] = df_candidates_report['rmse'].round(4)

# 위험 지표
df_candidates_report['리스크점수'] = df_candidates_report['risk_composite_raw'].round(3)
df_candidates_report['변동성'] = df_candidates_report['volatility'].round(4)
df_candidates_report['최대낙폭(%)'] = (df_candidates_report['max_drawdown'] * 100).round(2)

# 매매 정보
df_candidates_report['매수일'] = pd.to_datetime(df_candidates_report['buy_date']).dt.strftime('%Y-%m-%d')
df_candidates_report['매도일'] = pd.to_datetime(df_candidates_report['sell_date']).dt.strftime('%Y-%m-%d')
df_candidates_report['매수가'] = df_candidates_report['buy_price'].round(0).astype(int)
df_candidates_report['매도가'] = df_candidates_report['sell_price'].round(0).astype(int)

# 메타 정보
df_candidates_report['유동성점수'] = df_candidates_report['liquidity_score'].round(0).astype(int)

# ==========================================
# 3. 최종 리포트 컬럼 순서
# ==========================================
report_cols = [
    '순위', '티커', '종목명',
    
    # 수익성 (주요 지표)
    '예상일평균수익률(로그)', '예상총수익률(%)', '최적보유기간(일)',
    
    # 정확도 (2가지 지표)
    '방향성정확도(%)', '신뢰도(RMSE역수)', 'RMSE',
    
    # 위험
    '리스크점수', '변동성', '최대낙폭(%)',
    
    # 매매 정보
    '매수일', '매도일', '매수가', '매도가',
    
    # 메타
    '유동성점수'
]

df_report = df_candidates_report[report_cols]

# ==========================================
# 4. Top 20 미리보기
# ==========================================
print("\n📊 Top 20 종목 미리보기:")
print("\n[주요 지표 설명]")
print("   - 예상일평균수익률(로그): 시간당 복리 수익률 (높을수록 자본 효율 우수)")
print("   - 예상총수익률(%): 매수~매도 총 수익률")
print("   - 방향성정확도(%): 과거 예측의 상승/하락 방향 적중률")
print("   - 신뢰도(RMSE역수): 오차 크기 기반 신뢰도 (1에 가까울수록 정확)")
print("   - 리스크점수: 복합 위험 지표 (낮을수록 안전)")
print("   - 최적보유기간: 최대 수익을 위한 권장 보유일\n")

# 핵심 컬럼만 표시 (가독성)
display_cols_short = [
    '순위', '종목명', '예상총수익률(%)', '최적보유기간(일)',
    '방향성정확도(%)', '신뢰도(RMSE역수)', '리스크점수',
    '매수가', '매도가'
]

display(df_report[display_cols_short].head(20))

# ==========================================
# 5. 전체 후보 요약 통계
# ==========================================
print("\n📊 전체 후보 요약 통계:")

# 수익성 분포
print(f"\n[수익률 분포]")
print(f"   - 10% 이상: {(df_report['예상총수익률(%)'] >= 10).sum()}개")
print(f"   - 5~10%: {((df_report['예상총수익률(%)'] >= 5) & (df_report['예상총수익률(%)'] < 10)).sum()}개")
print(f"   - 0~5%: {((df_report['예상총수익률(%)'] >= 0) & (df_report['예상총수익률(%)'] < 5)).sum()}개")
print(f"   - 음수: {(df_report['예상총수익률(%)'] < 0).sum()}개")

# 보유기간 분포
print(f"\n[보유기간 분포]")
print(f"   - 5일 이하: {(df_report['최적보유기간(일)'] <= 5).sum()}개")
print(f"   - 6~10일: {((df_report['최적보유기간(일)'] > 5) & (df_report['최적보유기간(일)'] <= 10)).sum()}개")
print(f"   - 11~20일: {((df_report['최적보유기간(일)'] > 10) & (df_report['최적보유기간(일)'] <= 20)).sum()}개")
print(f"   - 21일 이상: {(df_report['최적보유기간(일)'] > 20).sum()}개")

# 정확도 분포
print(f"\n[정확도 분포]")
print(f"   - 방향성 70% 이상: {(df_report['방향성정확도(%)'] >= 70).sum()}개")
print(f"   - 방향성 60~70%: {((df_report['방향성정확도(%)'] >= 60) & (df_report['방향성정확도(%)'] < 70)).sum()}개")
print(f"   - 방향성 50~60%: {((df_report['방향성정확도(%)'] >= 50) & (df_report['방향성정확도(%)'] < 60)).sum()}개")
print(f"   - 신뢰도 0.7 이상: {(df_report['신뢰도(RMSE역수)'] >= 0.7).sum()}개")

# 위험도 분포
print(f"\n[위험도 분포]")
print(f"   - 낮은 리스크 (0~0.3): {(df_report['리스크점수'] <= 0.3).sum()}개")
print(f"   - 보통 리스크 (0.3~0.5): {((df_report['리스크점수'] > 0.3) & (df_report['리스크점수'] <= 0.5)).sum()}개")
print(f"   - 높은 리스크 (0.5~0.7): {((df_report['리스크점수'] > 0.5) & (df_report['리스크점수'] <= 0.7)).sum()}개")
print(f"   - 매우 높은 리스크 (0.7+): {(df_report['리스크점수'] > 0.7).sum()}개")

## 8️⃣ 결과 저장

In [ ]:
print("\n" + "="*65)
print("8️⃣ 결과 저장")
print("="*65)

# ==========================================
# 1. 전체 Universe (평가 완료)
# ==========================================
full_universe_path = output_dir / 'universe_full.parquet'
df_universe.to_parquet(full_universe_path, index=False)
print(f"\n💾 전체 Universe 저장: {full_universe_path}")
print(f"   - 종목 수: {len(df_universe):,}")

# ==========================================
# 2. Top-K 후보 (Parquet)
# ==========================================
candidates_path = output_dir / 'universe_candidates.parquet'
df_candidates.to_parquet(candidates_path, index=False)
print(f"\n💾 투자 후보 저장: {candidates_path}")
print(f"   - 종목 수: {len(df_candidates):,}")

# ==========================================
# 3. 상세 리포트 (CSV, 사람 가독성 우선)
# ==========================================
report_path = output_dir / 'investment_report.csv'
df_report.to_csv(report_path, index=False, encoding='utf-8-sig')
print(f"\n💾 상세 리포트 저장: {report_path}")
print(f"   - 형식: CSV (Excel 호환)")
print(f"   - 컬럼 수: {len(report_cols)}개")

# ==========================================
# 4. Excel용 요약 시트 (선택)
# ==========================================
try:
    excel_path = output_dir / 'investment_report.xlsx'
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Sheet 1: Top 20 요약
        df_report.head(20).to_excel(writer, sheet_name='Top20', index=False)
        
        # Sheet 2: 전체 후보
        df_report.to_excel(writer, sheet_name='전체후보', index=False)
        
        # Sheet 3: 위험별 분류
        df_risk_groups = df_report.copy()
        df_risk_groups['위험등급'] = pd.cut(
            df_risk_groups['리스크점수'],
            bins=[0, 0.3, 0.5, 0.7, 1.0],
            labels=['낮음', '보통', '높음', '매우높음']
        )
        
        for risk_level in ['낮음', '보통', '높음', '매우높음']:
            df_level = df_risk_groups[df_risk_groups['위험등급'] == risk_level]
            if len(df_level) > 0:
                df_level.to_excel(writer, sheet_name=f'위험_{risk_level}', index=False)
    
    print(f"\n💾 Excel 리포트 저장: {excel_path}")
    print(f"   - Sheet: Top20, 전체후보, 위험_낮음, 위험_보통 등")
    
except Exception as e:
    print(f"\n⚠️  Excel 저장 실패 (openpyxl 필요): {e}")

# ==========================================
# 5. 필터링 통계 저장
# ==========================================
filter_stats_path = output_dir / 'filter_statistics.json'

filter_summary = {
    'initial_count': initial_count,
    'final_count': len(df_candidates),
    'removal_count': initial_count - len(df_candidates),
    'removal_rate_%': (initial_count - len(df_candidates)) / initial_count * 100,
    
    'constraints': {
        'min_liquidity': MIN_LIQUIDITY,
        'max_risk': MAX_RISK,
        'max_accuracy_rank': 1000
    },
    
    'statistics': {
        'avg_expected_return_%': df_candidates['total_return_pct'].mean(),
        'avg_hold_days': df_candidates['hold_days'].mean(),
        'avg_directional_accuracy_%': df_candidates['directional_accuracy'].mean() * 100,
        'avg_rmse': df_candidates['rmse'].mean(),
        'avg_risk_score': df_candidates['risk_composite_raw'].mean()
    }
}

import json
with open(filter_stats_path, 'w', encoding='utf-8') as f:
    json.dump(filter_summary, f, indent=2, ensure_ascii=False)

print(f"\n💾 필터링 통계 저장: {filter_stats_path}")

print("\n" + "="*65)
print("✅ [Step 5] Universe 선정 완료")
print("="*65)
print(f"\n💡 다음 단계:")
print(f"   1. Excel/CSV 파일 열기: {report_path.name}")
print(f"   2. 수익률, 정확도, 위험을 종합 검토")
print(f"   3. 최종 투자 종목 수동 선택 (권장: 20~30개)")
print(f"   4. 선택한 종목으로 06단계 포트폴리오 최적화 진행")

## 🏁 완료 및 다음 단계

### ✅ 생성된 산출물
1. **`universe_candidates.parquet`**: 시스템이 선정한 Top-K 후보군 (데이터 처리용)
2. **`investment_report.csv` / `.xlsx`**: 사용자가 직접 보고 판단할 수 있는 **상세 분석 리포트** ⭐
3. **`filter_statistics.json`**: 필터링 단계별 제거 현황 로그

### 🎯 투자 후보 결정 가이드 (User Guide)
생성된 엑셀 리포트(`investment_report.xlsx`)를 열고 다음 순서로 검토하는 것을 권장합니다.

1. **상위 20위 검토**: 기대 수익률이 가장 높은 종목들입니다.
2. **신뢰도 교차 검증**:
   - **`방향성정확도(%)`**: 최소 60% 이상인가? (과거에 방향을 잘 맞췄는가?)
   - **`신뢰도(RMSE역수)`**: 값이 너무 낮지 않은가?
3. **리스크 확인**:
   - **`리스크점수`**: 0.7 이상이면 매우 위험하므로 투자 비중을 줄이거나 제외 고려.
   - **`최대낙폭(%)`**: 감당 가능한 수준인가?
4. **매매 계획**:
   - `최적보유기간`과 `매수/매도 목표가`를 참고하여 트레이딩 계획 수립.

### 🚀 다음 작업 (Next Step)
- **06단계**: 선정된 종목들을 대상으로 **포트폴리오 최적화 (MVO 등)**를 수행하여 최종 비중을 결정합니다.
- **백테스트**: 과거 구간에 대해 이 전략을 시뮬레이션하여 수익성을 검증합니다.